Jupyter Notebook for the Kaggle Competition (Machine Learning - CentraleSupélec - January 2025)  
PAES DE ALMEIDA NINA DUARTE, Pedro ; COLLIER, Grégoire ; PERARDT MAGALHÃES BRITO, Romero

Import necessary packages

In [68]:
import geopandas as gpd
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np


from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score

Import data (as shown in the baseline code) and treat NaN

In [69]:
change_type_map = {'Demolition': 0, 'Road': 1, 'Residential': 2, 'Commercial': 3, 'Industrial': 4,
                   'Mega Projects': 5}

## Read csvs

train_df = gpd.read_file('train.geojson', index_col=0)
test_df = gpd.read_file('test.geojson', index_col=0)

train_y = train_df['change_type'].apply(lambda x: change_type_map[x])

c:\Users\pedro\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GeoJSON does not support open option INDEX_COL
  return ogr_read(
c:\Users\pedro\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GeoJSON does not support open option INDEX_COL
  return ogr_read(


In [70]:
features = train_df.columns.to_list()
features = [feature for feature in features if feature != 'change_type']

In [71]:
train_df.shape

(296146, 45)

Fill NaN with mean

In [72]:
ignore_columns = ['urban_type', 'geography_type', 'change_type','date0', 'change_status_date0', 'date1',
                   'change_status_date1','date2', 'change_status_date2','date3', 'change_status_date3', 
                   'date4', 'change_status_date4', 'index', 'geometry']

for col in train_df.columns:
    if col not in ignore_columns:
        train_df[col].fillna(train_df[col].mean(), inplace=True)

C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\2416107534.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[col].fillna(train_df[col].mean(), inplace=True)


Fill missing dates with closest date (one above)

In [73]:
train_df['date0'].fillna(method='ffill', inplace=True)
train_df['date1'].fillna(method='ffill', inplace=True)
train_df['date2'].fillna(method='ffill', inplace=True)
train_df['date3'].fillna(method='ffill', inplace=True)
train_df['date4'].fillna(method='ffill', inplace=True)

C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\4175239564.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['date0'].fillna(method='ffill', inplace=True)
C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\4175239564.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train_df['date0'].fillna(method='ffill', inplace=True)
C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\4175239564.py:2: FutureWarning: A value is trying to be s

In [74]:
train_df.isna().any()

urban_type              False
geography_type          False
change_type             False
img_red_mean_date1      False
img_green_mean_date1    False
img_blue_mean_date1     False
img_red_std_date1       False
img_green_std_date1     False
img_blue_std_date1      False
img_red_mean_date2      False
img_green_mean_date2    False
img_blue_mean_date2     False
img_red_std_date2       False
img_green_std_date2     False
img_blue_std_date2      False
img_red_mean_date3      False
img_green_mean_date3    False
img_blue_mean_date3     False
img_red_std_date3       False
img_green_std_date3     False
img_blue_std_date3      False
img_red_mean_date4      False
img_green_mean_date4    False
img_blue_mean_date4     False
img_red_std_date4       False
img_green_std_date4     False
img_blue_std_date4      False
img_red_mean_date5      False
img_green_mean_date5    False
img_blue_mean_date5     False
img_red_std_date5       False
img_green_std_date5     False
img_blue_std_date5      False
date0     

Drop 1458 rows that had NaN in change_status columns

In [75]:
train_df.dropna(inplace=True)

Treat 'N,A' in 'urban_type' and 'geography_type' column

In [86]:
train_df[train_df['urban_type'] == 'N,A'].loc[train_df['geography_type'].str.contains("Dense Forest")]

,urban_type,geography_type,change_type,img_red_mean_date1,img_green_mean_date1,img_blue_mean_date1,img_red_std_date1,img_green_std_date1,img_blue_std_date1,img_red_mean_date2,...,date1,change_status_date1,date2,change_status_date2,date3,change_status_date3,date4,change_status_date4,index,geometry
81,"N,A",Dense Forest,Residential,79.559880,81.755639,74.073128,13.686379,11.910161,11.177457,58.669879,...,09-12-2013,Greenland,10-09-2016,Greenland,22-07-2019,Construction Done,24-07-2017,Land Cleared,81,"POLYGON ((112.16661 31.99345, 112.1668 31.9932..."
82,"N,A",Dense Forest,Residential,126.885399,127.487298,117.707654,42.231033,41.367180,35.573278,113.320185,...,09-12-2013,Prior Construction,10-09-2016,Prior Construction,22-07-2019,Greenland,24-07-2017,Prior Construction,82,"POLYGON ((112.16562 31.99285, 112.16591 31.992..."
83,"N,A",Dense Forest,Commercial,134.978603,136.923289,123.400654,39.776002,39.789677,33.242588,84.401066,...,09-12-2013,Operational,10-09-2016,Operational,22-07-2019,Construction Done,24-07-2017,Operational,83,"POLYGON ((112.16514 31.9919, 112.16553 31.9922..."
86,"N,A",Dense Forest,Road,96.225763,105.246082,85.399106,18.980312,15.461174,14.092705,96.912806,...,09-12-2013,Greenland,10-09-2016,Greenland,22-07-2019,Construction Midway,24-07-2017,Greenland,86,"POLYGON ((112.17227 31.993, 112.17323 31.99343..."
87,"N,A",Dense Forest,Road,97.518736,110.277567,90.296607,27.208074,25.422145,24.011413,98.709723,...,09-12-2013,Greenland,10-09-2016,Greenland,22-07-2019,Construction Midway,24-07-2017,Greenland,87,"POLYGON ((112.17334 31.99343, 112.17343 31.993..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296082,"N,A","Sparse Forest,Dense Forest,Lakes",Residential,241.472273,230.977727,211.636364,21.216292,22.192791,21.510051,115.922273,...,25-02-2017,Construction Done,27-01-2014,Land Cleared,28-03-2018,Construction Done,28-12-2015,Land Cleared,296082,"POLYGON ((-109.71339 23.03661, -109.7134 23.03..."
296083,"N,A","Sparse Forest,Dense Forest,Lakes",Residential,220.899960,224.273709,223.943177,43.616930,40.837289,41.547718,152.272509,...,25-02-2017,Construction Midway,27-01-2014,Prior Construction,28-03-2018,Construction Done,28-12-2015,Construction Midway,296083,"POLYGON ((-109.71393 23.03627, -109.71401 23.0..."
296091,"N,A","Sparse Forest,Dense Forest,Grass Land",Residential,152.487500,148.013542,138.698611,49.720446,58.123212,66.369411,122.394444,...,25-02-2017,Construction Midway,27-01-2014,Greenland,28-03-2018,Construction Done,28-12-2015,Greenland,296091,"POLYGON ((-109.71756 23.03663, -109.71752 23.0..."
296099,"N,A","Sparse Forest,Dense Forest,Grass Land",Residential,183.346243,175.017161,160.653641,62.405827,58.053958,61.766244,92.305891,...,25-02-2017,Construction Started,27-01-2014,Land Cleared,28-03-2018,Construction Done,28-12-2015,Land Cleared,296099,"POLYGON ((-109.71762 23.03392, -109.71753 23.0..."


In [83]:

train_df[train_df['urban_type'] == 'N,A'].loc[train_df['geography_type'].str.contains("Dense Forest"), 'urban_type'] = 'Rural'

C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\3107740386.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  train_df[train_df['urban_type'] == 'N,A'].loc[train_df['geography_type'].str.contains("Dense Forest"), 'urban_type'] = 'Rural'


In [77]:
train_df.head(20)

,urban_type,geography_type,change_type,img_red_mean_date1,img_green_mean_date1,img_blue_mean_date1,img_red_std_date1,img_green_std_date1,img_blue_std_date1,img_red_mean_date2,...,date1,change_status_date1,date2,change_status_date2,date3,change_status_date3,date4,change_status_date4,index,geometry
0,Sparse Urban,"Dense Forest,Grass Land",Road,93.371775,107.291113,89.827379,29.812040,28.328368,25.324294,125.773062,...,09-12-2013,Greenland,10-09-2016,Construction Started,22-07-2019,Construction Done,24-07-2017,Construction Midway,0,"POLYGON ((112.16774 32.02198, 112.16845 32.020..."
1,Sparse Urban,"Dense Forest,Grass Land",Road,96.071674,107.061702,90.755556,24.896240,22.275180,22.080686,133.097679,...,09-12-2013,Greenland,10-09-2016,Land Cleared,22-07-2019,Construction Done,24-07-2017,Construction Midway,1,"POLYGON ((112.16849 32.02048, 112.16891 32.019..."
2,Sparse Urban,"Dense Forest,Grass Land",Road,101.212148,113.462178,95.670574,24.179684,21.873401,21.285197,120.713490,...,09-12-2013,Greenland,10-09-2016,Land Cleared,22-07-2019,Construction Done,24-07-2017,Land Cleared,2,"POLYGON ((112.16892 32.01969, 112.16962 32.018..."
3,Rural,"Dense Forest,Grass Land",Road,94.463311,99.995531,84.470046,26.869852,23.767679,19.351983,114.819776,...,09-12-2013,Greenland,10-09-2016,Construction Started,22-07-2019,Construction Done,24-07-2017,Construction Midway,3,"POLYGON ((112.16966 32.0181, 112.17033 32.0166..."
4,Dense Urban,"Sparse Forest,Dense Forest,Farms",Demolition,151.883646,191.710197,211.569244,52.465332,59.441844,52.304349,141.514462,...,09-12-2013,Prior Construction,10-09-2016,Prior Construction,22-07-2019,Land Cleared,24-07-2017,Prior Construction,4,"POLYGON ((112.16669 32.01597, 112.16677 32.015..."
5,Sparse Urban,"Sparse Forest,Grass Land,Farms",Road,95.771293,106.029959,90.478691,22.555189,22.928741,21.705047,132.300162,...,09-12-2013,Greenland,10-09-2016,Land Cleared,22-07-2019,Construction Done,24-07-2017,Construction Midway,5,"POLYGON ((112.1704 32.01661, 112.17092 32.0155..."
6,"Urban Slum,Rural","Sparse Forest,Grass Land,Farms",Demolition,115.977543,152.283451,165.955321,44.652879,66.088330,81.148390,102.552152,...,09-12-2013,Prior Construction,10-09-2016,Prior Construction,22-07-2019,Land Cleared,24-07-2017,Prior Construction,6,"POLYGON ((112.16678 32.01433, 112.16687 32.014..."
7,Sparse Urban,"Sparse Forest,Grass Land,Farms",Road,103.210477,107.253144,103.452377,45.771980,44.858021,44.302871,119.425494,...,09-12-2013,Prior Construction,10-09-2016,Prior Construction,22-07-2019,Construction Done,24-07-2017,Prior Construction,7,"POLYGON ((112.171 32.01545, 112.17154 32.01437..."
8,Sparse Urban,"Sparse Forest,Grass Land,Farms",Road,89.780819,94.857300,85.456024,40.124317,37.900068,34.251561,84.645008,...,09-12-2013,Prior Construction,10-09-2016,Prior Construction,22-07-2019,Construction Done,24-07-2017,Prior Construction,8,"POLYGON ((112.17154 32.01428, 112.17224 32.012..."
9,Sparse Urban,"Dense Forest,Farms",Road,123.565988,130.994659,116.435717,56.816426,56.672889,49.360098,96.819217,...,09-12-2013,Land Cleared,10-09-2016,Operational,22-07-2019,Operational,24-07-2017,Construction Done,9,"POLYGON ((112.16566 32.0124, 112.16631 32.0107..."


In [84]:
train_df['urban_type'].value_counts()

urban_type
Dense Urban                           88975
Sparse Urban                          68877
Industrial                            60058
N,A                                   36545
Rural                                 20356
Sparse Urban,Industrial                8076
Dense Urban,Industrial                 7177
Sparse Urban,Urban Slum                1938
Urban Slum                             1468
Dense Urban,Urban Slum                  664
Rural,Industrial                        188
Urban Slum,Industrial                   179
Sparse Urban,Dense Urban                 72
Dense Urban,Rural                        52
Sparse Urban,Urban Slum,Industrial       47
Sparse Urban,Rural                       20
Urban Slum,Rural                          6
Name: count, dtype: int64

In [79]:
train_df[train_df.geography_type == 'N,A'].shape[0]

6153

In [80]:
train_df.shape

(294698, 45)